# Phase 5h — Relevance Sensitivity for the Main System (MuQ + SBERT)

Phase 5c tested graded relevance (nDCG with σ ∈ {0.05, 0.10, 0.20}) and thresholded AV relevance (P@10(τ)) for the
CLAP-based system only. The revised manuscript defines both for the main system, so this notebook repeats the Phase 5c
metrics for the Phase 5f configurations (MuQ L6 z, SBERT z, late fusion α = 0.6, early fusion ≡ α = 0.5; all chosen on
validation in 5e/5f). Shared splits and catalog as in 5f; the σ = 0.10 nDCG and quadrant P@10 are asserted equal to
`main_results_table.csv`.

Paired tests (Wilcoxon–Pratt, paired bootstrap 95% CI, Holm within split × metric): late fusion − lyrics-only,
late fusion − audio-only, audio-only − lyrics-only.

Output: `relevance_sensitivity_main.csv`, `relevance_sensitivity_main_tests.csv`. AV values are evaluation targets only.

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.preprocessing import StandardScaler

ROOT = os.environ.get('MUSIC_RESEARCH_ROOT', os.path.abspath('.'))
OUT_DIR = os.path.join(ROOT, 'outputs')
ENC_DIR = os.path.join(OUT_DIR, 'encoders')
K, N_BOOT, SEED = 10, 10_000, 42
SIGMAS = [0.05, 0.10, 0.20]
TAUS   = [0.05, 0.075, 0.10, 0.15, 0.20, 0.25, 0.30]
SCHEMES = {'tvt_70_15_15': '70/15/15', 'tvt_40_30_30': '40/30/30'}
rng = np.random.default_rng(SEED)

lyrics_df = pd.read_parquet(os.path.join(OUT_DIR, 'lyrics_features.parquet'))
merged    = pd.read_parquet(os.path.join(OUT_DIR, 'merged_dataset.parquet'))
cat_sids  = lyrics_df['song_id'].tolist()
cat_pos   = {s: i for i, s in enumerate(cat_sids)}
assert pd.read_csv(os.path.join(ENC_DIR, 'text_song_ids.csv'))['song_id'].tolist() == cat_sids
a_ids = pd.read_csv(os.path.join(ENC_DIR, 'audio_song_ids.csv'))['song_id'].tolist()
a_row = [{s: i for i, s in enumerate(a_ids)}[s] for s in cat_sids]

def zunit(X):
    X = StandardScaler().fit_transform(X.astype(np.float64))
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
A = zunit(np.load(os.path.join(ENC_DIR, 'audio_muq.npy'), mmap_mode='r')[a_row, 6])
T = zunit(np.load(os.path.join(ENC_DIR, 'text_sbert_mpnet_trunc.npy')))

lab  = merged.set_index('song_id').loc[cat_sids]
quad = lab['quadrant'].values
Y    = lab[['arousal', 'valence']].values.astype(np.float64)
D_AV = np.linalg.norm(Y[:, None] - Y[None], axis=-1)

mm = merged[merged['is_multimodal']]
lyr_stem_to_sid = {os.path.splitext(os.path.basename(p))[0]: s for s, p in zip(mm['song_id'], mm['lyrics_path'])}
def read_split(modality, scheme, subset):
    folder = 'MERGE_Audio_Balanced' if modality == 'audio' else 'MERGE_Lyrics_Balanced'
    ids = pd.read_csv(os.path.join(ROOT, folder, 'tvt_dataframes', scheme,
                                   f'{scheme}_{subset}_{modality}_balanced.csv'))['Song']
    return set(ids.map(lyr_stem_to_sid).dropna()) if modality == 'lyrics' else set(ids)
cat = set(cat_sids)
test = {lbl: sorted(read_split('audio', sc, 'test') & read_split('lyrics', sc, 'test') & cat) for sc, lbl in SCHEMES.items()}
print({k: len(v) for k, v in test.items()})

STRATS = {'audio-only (MuQ)': lambda r: A[r] @ A.T, 'lyrics-only (SBERT)': lambda r: T[r] @ T.T,
          'late fusion (α=0.6)': lambda r: 0.6 * (A[r] @ A.T) + 0.4 * (T[r] @ T.T),
          'early fusion (≡ α=0.5)': lambda r: 0.5 * (A[r] @ A.T) + 0.5 * (T[r] @ T.T)}
DISC = 1 / np.log2(np.arange(2, K + 2))
def topk(S, k):
    top = np.argpartition(-S, k, 1)[:, :k]
    return np.take_along_axis(top, np.take_along_axis(S, top, 1).argsort(1)[:, ::-1], 1)

PER = {}
for lbl, qs in test.items():
    rows = np.array([cat_pos[s] for s in qs])
    Dq = D_AV[rows].copy(); Dq[np.arange(len(rows)), rows] = np.inf
    d_ideal = np.sort(Dq, 1)[:, :K]
    others = np.isfinite(Dq)
    chance = {t: ((Dq <= t) & others).sum(1) / others.sum(1) for t in TAUS}
    for name, f in STRATS.items():
        S = f(rows); S[np.arange(len(rows)), rows] = -np.inf
        top = topk(S, K); d = np.take_along_axis(D_AV[rows], top, 1)
        m = {'P@10 (quadrant)': (quad[top] == quad[rows][:, None]).mean(1)}
        for s in SIGMAS:
            g, gi = np.exp(-d**2 / (2 * s**2)), np.exp(-d_ideal**2 / (2 * s**2))
            m[f'nDCG@10 σ={s}'] = (g * DISC).sum(1) / (gi * DISC).sum(1)
        for t in TAUS:
            m[f'P@10(τ={t})'] = (d <= t).mean(1)
        PER[(lbl, name)] = m
    PER[(lbl, 'chance')] = {f'P@10(τ={t})': chance[t] for t in TAUS} | {'P@10 (quadrant)': np.full(len(rows), np.nan)}

ref = pd.read_csv(os.path.join(OUT_DIR, 'main_results_table.csv'))
for lbl in test:
    r = ref[ref.split == lbl].set_index('strategy')
    for name in STRATS:
        assert abs(PER[(lbl, name)]['P@10 (quadrant)'].mean() - r.loc[name, 'P@10']) < 1e-9, (lbl, name)
        assert abs(PER[(lbl, name)]['nDCG@10 σ=0.1'].mean() - r.loc[name, 'nDCG@10']) < 1e-9, (lbl, name)
print('Quadrant P@10 and nDCG@10 (σ = 0.10) match main_results_table.csv.')

{'70/15/15': 302, '40/30/30': 617}


Quadrant P@10 and nDCG@10 (σ = 0.10) match main_results_table.csv.


In [2]:
def boot_ci(x):
    idx = rng.integers(0, len(x), (N_BOOT, len(x)))
    return np.percentile(x[idx].mean(1), [2.5, 97.5])

rows = []
for (lbl, name), m in PER.items():
    for metric, x in m.items():
        if np.isnan(x).all(): continue
        lo, hi = boot_ci(x) if name != 'chance' else (np.nan, np.nan)
        rows.append({'split': lbl, 'strategy': name, 'metric': metric, 'mean': x.mean(), 'lo': lo, 'hi': hi})
summ = pd.DataFrame(rows)
summ.to_csv(os.path.join(OUT_DIR, 'relevance_sensitivity_main.csv'), index=False)
for lbl in test:
    print(f'\n=== {lbl} test (n={len(test[lbl])}) ===')
    print(summ[summ.split == lbl].pivot(index='metric', columns='strategy', values='mean')
          [list(STRATS) + ['chance']].round(3).to_string())


=== 70/15/15 test (n=302) ===
strategy         audio-only (MuQ)  lyrics-only (SBERT)  late fusion (α=0.6)  early fusion (≡ α=0.5)  chance
metric                                                                                                     
P@10 (quadrant)             0.625                0.605                0.702                   0.704     NaN
P@10(τ=0.05)                0.203                0.174                0.236                   0.227   0.058
P@10(τ=0.075)               0.313                0.275                0.347                   0.335   0.102
P@10(τ=0.1)                 0.405                0.377                0.453                   0.447   0.146
P@10(τ=0.15)                0.531                0.519                0.597                   0.599   0.226
P@10(τ=0.2)                 0.599                0.613                0.679                   0.681   0.306
P@10(τ=0.25)                0.650                0.677                0.732                   0.735   0.3

In [3]:
def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1.0, run)
    return adj

pairs = [('late fusion (α=0.6)', 'lyrics-only (SBERT)'), ('late fusion (α=0.6)', 'audio-only (MuQ)'),
         ('audio-only (MuQ)', 'lyrics-only (SBERT)')]
metrics = [f'nDCG@10 σ={s}' for s in SIGMAS] + [f'P@10(τ={t})' for t in TAUS]
trows = []
for lbl in test:
    for metric in metrics:
        block = []
        for a, b in pairs:
            xa, xb = PER[(lbl, a)][metric], PER[(lbl, b)][metric]; d = xa - xb; lo, hi = boot_ci(d)
            p = wilcoxon(xa, xb, zero_method='pratt').pvalue if (d != 0).any() else 1.0
            block.append({'split': lbl, 'metric': metric, 'comparison': f'{a} − {b}', 'diff': d.mean(), 'lo': lo, 'hi': hi, 'p': p})
        for r, ph in zip(block, holm([r['p'] for r in block])):
            r['p_holm'] = ph
        trows += block
tests = pd.DataFrame(trows)
tests.to_csv(os.path.join(OUT_DIR, 'relevance_sensitivity_main_tests.csv'), index=False)
print(tests.to_string(index=False, float_format=lambda x: f'{x:.3g}'))

   split         metric                                comparison     diff        lo       hi        p   p_holm
70/15/15 nDCG@10 σ=0.05 late fusion (α=0.6) − lyrics-only (SBERT)   0.0552    0.0388   0.0714 2.97e-09  8.9e-09
70/15/15 nDCG@10 σ=0.05    late fusion (α=0.6) − audio-only (MuQ)   0.0261     0.015   0.0375  6.9e-05 0.000138
70/15/15 nDCG@10 σ=0.05    audio-only (MuQ) − lyrics-only (SBERT)   0.0291   0.00952   0.0489    0.011    0.011
70/15/15  nDCG@10 σ=0.1 late fusion (α=0.6) − lyrics-only (SBERT)   0.0634    0.0438   0.0842 5.22e-09 1.04e-08
70/15/15  nDCG@10 σ=0.1    late fusion (α=0.6) − audio-only (MuQ)   0.0498    0.0353   0.0646  5.6e-10 1.68e-09
70/15/15  nDCG@10 σ=0.1    audio-only (MuQ) − lyrics-only (SBERT)   0.0136   -0.0117   0.0386    0.463    0.463
70/15/15  nDCG@10 σ=0.2 late fusion (α=0.6) − lyrics-only (SBERT)   0.0484    0.0286   0.0695 3.13e-06 6.26e-06
70/15/15  nDCG@10 σ=0.2    late fusion (α=0.6) − audio-only (MuQ)   0.0583     0.043   0.0741 8.88e-12 2